# YOLO26 — Lab Rumiología (Colab)

1. **Runtime → Change runtime type → GPU**
2. Genere el ZIP en su PC: `python ml/scripts/package_colab_dataset.py`
3. Suba `ml/exports/rumiologia_yolo_dataset.zip` a `/content/` (o Drive)
4. Ejecute las celdas
5. Descargue `best.pt`, `model.tflite`, `labels.txt`
6. En el PC: `python ml/scripts/integrate_colab_artifacts.py --dir <descargas>`

Alineado con `ml/scripts/train_yolo.py` y `export_tflite.py`.

In [ ]:
!pip install -q ultralytics pyyaml pillow

In [ ]:
from google.colab import files
from pathlib import Path

DATA_ZIP = Path("/content/rumiologia_yolo_dataset.zip")
if not DATA_ZIP.exists():
    print("Seleccione rumiologia_yolo_dataset.zip…")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            DATA_ZIP.write_bytes(uploaded[name])
            break
print("ZIP:", DATA_ZIP, DATA_ZIP.exists())

In [ ]:
from pathlib import Path
import shutil
import zipfile

import yaml
from ultralytics import YOLO

DATA_ZIP = Path("/content/rumiologia_yolo_dataset.zip")
WORK = Path("/content/rumiologia")
MODELS = WORK / "models"
RUNS = WORK / "runs"
MODELS.mkdir(parents=True, exist_ok=True)

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP, "r") as zf:
    zf.extractall(WORK)

data_yaml = WORK / "data.yaml"
cfg = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
cfg["path"] = str(WORK)
runtime = WORK / "_runtime_data.yaml"
runtime.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Clases:", cfg.get("names"))
print("Train images:", len(list((WORK / "yolo_ls" / "images" / "train").glob("*"))))

In [ ]:
model = None
for name in ("yolo26m.pt", "yolo26s.pt", "yolo26n.pt", "yolov8n.pt"):
    try:
        model = YOLO(name)
        print("Base:", name)
        break
    except Exception as exc:
        print(name, "->", exc)
assert model is not None

results = model.train(
    data=str(runtime),
    epochs=120,
    imgsz=640,
    batch=-1,
    project=str(RUNS),
    name="rumiologia",
    exist_ok=True,
    patience=40,
    close_mosaic=15,
    cos_lr=True,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=3.0,
    translate=0.10,
    scale=0.30,
    fliplr=0.5,
    mosaic=0.5,
    workers=2,
    plots=True,
)

best = Path(results.save_dir) / "weights" / "best.pt"
(MODELS / "best.pt").write_bytes(best.read_bytes())
print("Guardado:", MODELS / "best.pt")
print(model.val(data=str(runtime), split="test"))

In [ ]:
try:
    exported = Path(model.export(format="tflite", imgsz=640, half=True, nms=False, end2end=False))
except Exception as e:
    print("half falló:", e)
    exported = Path(model.export(format="tflite", imgsz=640, nms=False, end2end=False))

tflite_out = MODELS / "model.tflite"
shutil.copy2(exported, tflite_out)

names = cfg.get("names", {})
if isinstance(names, dict):
    ordered = [names[i] for i in sorted(names, key=lambda k: int(k))]
else:
    ordered = list(names)
(MODELS / "labels.txt").write_text("\n".join(ordered) + "\n", encoding="utf-8")
print("Artefactos en", MODELS)
print(list(MODELS.iterdir()))

In [ ]:
from google.colab import files
for name in ("best.pt", "model.tflite", "labels.txt"):
    path = MODELS / name
    if path.exists():
        files.download(str(path))